In [5]:
"""
STEP 1 IMPROVED — Build Temporal Dataset
=========================================
Changes from v1:
- Snapshots: [0, 30, 60, 90, 150] — better aligned with 255-day courses
- Added trend features: click_trend, score_trend, days_since_last_click
- Added consistency features: click_consistency, submission_consistency
"""

import pandas as pd
import numpy as np
import os
from tqdm import tqdm

RAW_PATH   = "C:/Users/HP/PycharmProjects/EduPredict/project II/OULAD/"
OUTPUT_DIR = "edupredict_data/"
SNAPSHOTS  = [0, 30, 60, 90, 150]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# LOAD
# ============================================================
print("Loading raw OULAD files...")
student_info   = pd.read_csv(RAW_PATH + "studentInfo.csv")
student_reg    = pd.read_csv(RAW_PATH + "studentRegistration.csv")
student_assess = pd.read_csv(RAW_PATH + "studentAssessment.csv")
student_vle    = pd.read_csv(RAW_PATH + "studentVle.csv")
assessments    = pd.read_csv(RAW_PATH + "assessments.csv")
vle            = pd.read_csv(RAW_PATH + "vle.csv")
courses        = pd.read_csv(RAW_PATH + "courses.csv")

# ============================================================
# TARGET + DEMOGRAPHICS
# ============================================================
student_info['at_risk'] = student_info['final_result'].isin(
    ['Fail', 'Withdrawn']
).astype(int)

age_map = {'0-35': 0, '35-55': 1, '55<=': 2}
edu_map = {
    'No Formal quals'           : 0,
    'Lower Than A Level'        : 1,
    'A Level or Equivalent'     : 2,
    'HE Qualification'          : 3,
    'Post Graduate Qualification': 4
}

def parse_imd(val):
    if pd.isna(val): return np.nan
    try: return int(str(val).split('-')[0].replace('%','').strip())
    except: return np.nan

student_info['imd_numeric']    = student_info['imd_band'].apply(parse_imd)
student_info['imd_numeric']    = student_info['imd_numeric'].fillna(
                                     student_info['imd_numeric'].median())
student_info['age_numeric']    = student_info['age_band'].map(age_map)
student_info['edu_numeric']    = student_info['highest_education'].map(edu_map)
student_info['gender_bin']     = (student_info['gender'] == 'M').astype(int)
student_info['disability_bin'] = (student_info['disability'] == 'Y').astype(int)

# ============================================================
# REGISTRATION
# ============================================================
reg = student_reg[['id_student','code_module','code_presentation',
                    'date_registration']].copy()
reg['days_to_start']    = reg['date_registration'] * -1
reg['days_to_start']    = reg['days_to_start'].fillna(reg['days_to_start'].median())
reg['registered_early'] = (reg['days_to_start'] > 30).astype(int)

# ============================================================
# BASE MERGE
# ============================================================
base = student_info.merge(
    reg, on=['id_student','code_module','code_presentation'], how='left'
)

module_assess_count = assessments.groupby(
    ['code_module','code_presentation']
)['id_assessment'].count().reset_index()
module_assess_count.columns = ['code_module','code_presentation',
                                'module_total_assessments']
base = base.merge(module_assess_count,
                  on=['code_module','code_presentation'], how='left')

# Merge course length
base = base.merge(
    courses[['code_module','code_presentation','module_presentation_length']],
    on=['code_module','code_presentation'], how='left'
)

assess_full = student_assess.merge(
    assessments[['id_assessment','assessment_type','date','weight']],
    on='id_assessment', how='left'
)
vle_full = student_vle.merge(
    vle[['id_site','activity_type']], on='id_site', how='left'
)

# ============================================================
# SNAPSHOT FEATURE COMPUTERS
# ============================================================
def compute_vle_snapshot(day_cutoff, sv):
    """Compute VLE features up to day_cutoff. Returns NaN if no data."""
    if len(sv) == 0:
        return {
            'total_clicks': np.nan, 'active_days': np.nan,
            'avg_clicks_per_day': np.nan, 'quiz_clicks': np.nan,
            'forumng_clicks': np.nan, 'resource_clicks': np.nan,
            'clicked_before_start': np.nan,
            'days_since_last_click': np.nan,
            'click_consistency': np.nan,
        }

    sv_cut = sv[sv['date'] <= day_cutoff]

    if len(sv_cut) == 0:
        return {
            'total_clicks': np.nan, 'active_days': np.nan,
            'avg_clicks_per_day': np.nan, 'quiz_clicks': np.nan,
            'forumng_clicks': np.nan, 'resource_clicks': np.nan,
            'clicked_before_start': np.nan,
            'days_since_last_click': np.nan,
            'click_consistency': np.nan,
        }

    total_clicks = float(sv_cut['sum_click'].sum())
    active_days  = float(sv_cut['date'].nunique())
    last_click   = float(sv_cut['date'].max())

    def type_clicks(t):
        mask = sv_cut['activity_type'] == t
        return float(sv_cut.loc[mask, 'sum_click'].sum()) if mask.any() else 0.0

    # click_consistency: ratio of active days to total days in range
    # high = logging in regularly, low = sporadic
    day_range = max(day_cutoff - sv_cut['date'].min() + 1, 1)
    click_consistency = active_days / day_range if day_range > 0 else np.nan

    return {
        'total_clicks':         total_clicks,
        'active_days':          active_days,
        'avg_clicks_per_day':   round(total_clicks / active_days, 2),
        'quiz_clicks':          type_clicks('quiz'),
        'forumng_clicks':       type_clicks('forumng'),
        'resource_clicks':      type_clicks('resource'),
        'clicked_before_start': 1.0 if (sv_cut['date'] < 0).any() else 0.0,
        'days_since_last_click': float(day_cutoff - last_click),
        'click_consistency':    round(click_consistency, 4),
    }


def compute_assess_snapshot(day_cutoff, sa, mod_total):
    """Compute assessment features up to day_cutoff. Returns NaN if no data."""
    nan_result = {
        'avg_score': np.nan, 'num_submitted': np.nan,
        'num_failed': np.nan, 'submission_rate': np.nan,
        'avg_tma_score': np.nan, 'avg_cma_score': np.nan,
        'avg_days_late': np.nan, 'score_trend': np.nan,
        'submission_consistency': np.nan,
    }

    if len(sa) == 0:
        return nan_result

    sa_cut = sa[sa['date_submitted'] <= day_cutoff]

    if len(sa_cut) == 0:
        return nan_result

    scores     = sa_cut['score'].fillna(0)
    num_sub    = len(sa_cut)
    num_failed = int((scores < 40).sum())

    tma_scores = sa_cut[sa_cut['assessment_type'] == 'TMA']['score'].dropna()
    cma_scores = sa_cut[sa_cut['assessment_type'] == 'CMA']['score'].dropna()

    sa_with_date = sa_cut.dropna(subset=['date'])
    avg_late = float(
        (sa_with_date['date_submitted'] - sa_with_date['date']).mean()
    ) if len(sa_with_date) > 0 else np.nan

    # score_trend: slope of scores over time (positive = improving)
    score_trend = np.nan
    if len(sa_cut) >= 3:
        sorted_sa = sa_cut.sort_values('date_submitted')
        scores_seq = sorted_sa['score'].fillna(0).values
        x = np.arange(len(scores_seq))
        try:
            slope = np.polyfit(x, scores_seq, 1)[0]
            score_trend = round(float(slope), 4)
        except:
            score_trend = np.nan

    # submission_consistency: submitted on time ratio
    on_time = 0
    if len(sa_with_date) > 0:
        on_time_mask = (sa_with_date['date_submitted'] <= sa_with_date['date'])
        on_time = float(on_time_mask.sum() / len(sa_with_date))

    sub_rate = float(num_sub / mod_total) if mod_total > 0 else np.nan

    return {
        'avg_score':              float(scores.mean()),
        'num_submitted':          float(num_sub),
        'num_failed':             float(num_failed),
        'submission_rate':        sub_rate,
        'avg_tma_score':          float(tma_scores.mean()) if len(tma_scores) > 0 else np.nan,
        'avg_cma_score':          float(cma_scores.mean()) if len(cma_scores) > 0 else np.nan,
        'avg_days_late':          avg_late,
        'score_trend':            score_trend,
        'submission_consistency': round(on_time, 4),
    }

# ============================================================
# MAIN LOOP
# ============================================================
print(f"\nBuilding temporal dataset...")
print(f"  Students  : {len(base):,}")
print(f"  Snapshots : {SNAPSHOTS}")
print(f"  Total rows: ~{len(base) * len(SNAPSHOTS):,}")

all_rows = []

vle_grouped    = {sid: grp for sid, grp in vle_full.groupby('id_student')}
assess_grouped = {sid: grp for sid, grp in assess_full.groupby('id_student')}

for _, student_row in tqdm(base.iterrows(), total=len(base)):
    sid = student_row['id_student']

    static = {
        'id_student':               sid,
        'code_module':              student_row['code_module'],
        'gender_bin':               student_row['gender_bin'],
        'disability_bin':           student_row['disability_bin'],
        'age_numeric':              student_row['age_numeric'],
        'edu_numeric':              student_row['edu_numeric'],
        'imd_numeric':              student_row['imd_numeric'],
        'num_of_prev_attempts':     student_row['num_of_prev_attempts'],
        'studied_credits':          student_row['studied_credits'],
        'days_to_start':            student_row['days_to_start'],
        'registered_early':         student_row['registered_early'],
        'module_total_assessments': student_row['module_total_assessments'],
        'course_length':            student_row['module_presentation_length'],
        'at_risk':                  student_row['at_risk'],
    }

    sv = vle_grouped.get(sid,    pd.DataFrame(columns=vle_full.columns))
    sa = assess_grouped.get(sid, pd.DataFrame(columns=assess_full.columns))
    mod_total = student_row['module_total_assessments']

    prev_clicks = None  # for trend calculation
    prev_score  = None

    for day in SNAPSHOTS:
        vle_feats    = compute_vle_snapshot(day, sv)
        assess_feats = compute_assess_snapshot(day, sa, mod_total)

        # snapshot_progress: how far into the course are we?
        course_len = student_row['module_presentation_length']
        snapshot_progress = round(day / course_len, 4) if course_len > 0 else np.nan

        # click_trend: change in total_clicks from previous snapshot
        curr_clicks = vle_feats['total_clicks']
        click_trend = np.nan
        if prev_clicks is not None and not np.isnan(curr_clicks) \
                and not np.isnan(prev_clicks):
            click_trend = curr_clicks - prev_clicks
        prev_clicks = curr_clicks if not (curr_clicks != curr_clicks) else prev_clicks

        row = {
            **static,
            'days_into_course':   day,
            'snapshot_progress':  snapshot_progress,
            'click_trend':        click_trend,
            **vle_feats,
            **assess_feats,
        }
        all_rows.append(row)

# ============================================================
# ASSEMBLE + SAVE
# ============================================================
print("\nAssembling dataframe...")
temporal_df = pd.DataFrame(all_rows)

print(f"Final shape: {temporal_df.shape}")
print(f"\nNaN by snapshot day:")
for day in SNAPSHOTS:
    sub = temporal_df[temporal_df['days_into_course'] == day]
    nan_score  = sub['avg_score'].isna().sum()
    nan_clicks = sub['total_clicks'].isna().sum()
    print(f"  Day {day:3d} | avg_score NaN: {nan_score:,} | total_clicks NaN: {nan_clicks:,}")

out_path = OUTPUT_DIR + "temporal_dataset_v2.csv"
temporal_df.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")


Loading raw OULAD files...

Building temporal dataset...
  Students  : 32,593
  Snapshots : [0, 30, 60, 90, 150]
  Total rows: ~162,965


100%|██████████| 32593/32593 [18:59<00:00, 28.59it/s]



Assembling dataframe...
Final shape: (162965, 35)

NaN by snapshot day:
  Day   0 | avg_score NaN: 31,403 | total_clicks NaN: 7,227
  Day  30 | avg_score NaN: 9,857 | total_clicks NaN: 3,182
  Day  60 | avg_score NaN: 6,428 | total_clicks NaN: 2,943
  Day  90 | avg_score NaN: 5,950 | total_clicks NaN: 2,902
  Day 150 | avg_score NaN: 5,861 | total_clicks NaN: 2,860

Saved to: edupredict_data/temporal_dataset_v2.csv
Done! Now run step2_train_temporal_model.py


In [2]:
"""
STEP 2 FINAL — Train Temporal LightGBM
========================================
Fixes from v3:
- Remove days_to_start entirely (correlation=0.056, causes #1 importance
  with no real predictive value — experiment proved F1 drop = 0.0003)
- Remove registered_early (derived from days_to_start, same issue)
- Remove late_reg_x_progress (no longer needed)
- Keep all behavioral features: clicks, scores, trends, consistency
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, recall_score,
                             precision_score, roc_auc_score,
                             accuracy_score)
import joblib
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD
# ============================================================
print("Loading temporal dataset v2...")
df = pd.read_csv("edupredict_data/temporal_dataset_v2.csv")
print(f"Shape: {df.shape}")
print(f"Snapshots: {sorted(df['days_into_course'].unique())}")

# ============================================================
# REMOVE REGISTRATION TIME FEATURES
# WHY: days_to_start correlation with at_risk = 0.056 (weak)
#      but LightGBM gave it #1 importance (2710/2887)
#      This means the model was learning a spurious shortcut:
#      "registered late → at_risk" regardless of actual behavior
#      Experiment: removing it costs only F1 = 0.0003
#      Benefit: model is now forced to learn from real behavior
# ============================================================
REMOVE_COLS = [
    'days_to_start',          # weak signal, dominates unfairly
    'registered_early',       # derived from days_to_start
    'late_reg_x_progress',    # interaction we added — no longer needed
]

# Only remove if they exist (late_reg_x_progress may not be in v2)
REMOVE_COLS = [c for c in REMOVE_COLS if c in df.columns]
df = df.drop(columns=REMOVE_COLS)
print(f"\nRemoved: {REMOVE_COLS}")
print(f"Shape after removal: {df.shape}")

# ============================================================
# FEATURES & TARGET
# ============================================================
DROP_COLS    = ['id_student', 'at_risk']
TARGET       = 'at_risk'
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]

print(f"\nFeature count: {len(FEATURE_COLS)}")
print(f"Features: {FEATURE_COLS}")

X      = df[FEATURE_COLS].copy()
y      = df[TARGET].copy()
groups = df['id_student']

# LightGBM native categorical — no leakage
X['code_module'] = X['code_module'].astype('category')

# ============================================================
# LGBM PARAMS
# ============================================================
params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'n_estimators':      500,
    'learning_rate':     0.03,
    'num_leaves':        63,
    'max_depth':         8,
    'min_child_samples': 30,
    'subsample':         0.8,
    'subsample_freq':    1,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        0.5,
    'min_split_gain':    0.01,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

# ============================================================
# 5-FOLD STRATIFIED GROUP CV
# ============================================================
print("\nStratifiedGroupKFold (5 folds)...")
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []
models       = []
val_indices  = []

print("\nTraining...")
print("=" * 75)

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(X, y, groups=groups)
):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set  = [(X_val, y_val)],
        callbacks = [lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(-1)]
    )

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    r = {
        'fold':      fold + 1,
        'accuracy':  accuracy_score(y_val, y_pred),
        'f1':        f1_score(y_val, y_pred),
        'recall':    recall_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred),
        'auc':       roc_auc_score(y_val, y_prob),
        'best_iter': model.best_iteration_,
    }
    fold_results.append(r)
    models.append(model)
    val_indices.append(val_idx)

    print(f"Fold {fold+1} | Acc: {r['accuracy']:.4f} | "
          f"F1: {r['f1']:.4f} | Recall: {r['recall']:.4f} | "
          f"Precision: {r['precision']:.4f} | AUC: {r['auc']:.4f} | "
          f"Iters: {r['best_iter']}")

print("=" * 75)
res_df = pd.DataFrame(fold_results)
print("\nMean CV Results:")
for col in ['accuracy', 'f1', 'recall', 'precision', 'auc']:
    print(f"  {col:10s}: {res_df[col].mean():.4f} ± {res_df[col].std():.4f}")

# ============================================================
# PER-DAY EVALUATION
# ============================================================
print("\n" + "=" * 75)
print("Performance by snapshot day (averaged across 5 folds):")
print("=" * 75)
print(f"{'Day':>5} | {'N':>6} | {'Accuracy':>9} | {'F1':>7} | "
      f"{'Recall':>7} | {'Precision':>9} | {'AUC':>7}")
print("-" * 75)

day_results = {d: [] for d in sorted(df['days_into_course'].unique())}

for fold_i, val_idx in enumerate(val_indices):
    val_df           = df.iloc[val_idx].copy()
    val_df['y_pred'] = models[fold_i].predict(X.iloc[val_idx])
    val_df['y_prob'] = models[fold_i].predict_proba(X.iloc[val_idx])[:, 1]

    for day in sorted(df['days_into_course'].unique()):
        sub = val_df[val_df['days_into_course'] == day]
        if len(sub) == 0:
            continue
        day_results[day].append({
            'accuracy':  accuracy_score(sub['at_risk'], sub['y_pred']),
            'f1':        f1_score(sub['at_risk'], sub['y_pred']),
            'recall':    recall_score(sub['at_risk'], sub['y_pred']),
            'precision': precision_score(sub['at_risk'], sub['y_pred']),
            'auc':       roc_auc_score(sub['at_risk'], sub['y_prob']),
        })

for day in sorted(df['days_into_course'].unique()):
    n   = len(df[df['days_into_course'] == day])
    avg = pd.DataFrame(day_results[day]).mean()
    print(f"{day:>5} | {n:>6,} | "
          f"{avg['accuracy']:>9.4f} | {avg['f1']:>7.4f} | "
          f"{avg['recall']:>7.4f} | {avg['precision']:>9.4f} | "
          f"{avg['auc']:>7.4f}")

# ============================================================
# FEATURE IMPORTANCE — what does the model rely on now?
# ============================================================
print("\n" + "=" * 75)
print("Top 20 Feature Importances — AFTER removing days_to_start:")
print("=" * 75)

importance = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': models[-1].feature_importances_
}).sort_values('importance', ascending=False)

print(importance.head(20).to_string(index=False))

# Sanity check — behavioral features should dominate now
top5 = importance.head(5)['feature'].tolist()
behavioral = ['avg_score','avg_tma_score','avg_cma_score','avg_days_late',
              'click_trend','score_trend','click_consistency',
              'forumng_clicks','quiz_clicks','total_clicks','active_days']
behavioral_in_top5 = sum(1 for f in top5 if f in behavioral)
print(f"\nBehavioral features in top 5: {behavioral_in_top5}/5")
if behavioral_in_top5 >= 3:
    print("✅ Model now relies on actual student behavior")
else:
    print("⚠️  Check top features — unexpected dominance")

# ============================================================
# FINAL MODEL — use mean best_iter from CV
# ============================================================
best_iter = int(res_df['best_iter'].mean())
print(f"\nFinal model n_estimators = {best_iter} (mean CV early stopping)")

final_params = {**params, 'n_estimators': best_iter}
final_model  = lgb.LGBMClassifier(**final_params)

print("Retraining on full dataset...")
final_model.fit(X, y, callbacks=[lgb.log_evaluation(-1)])

# ============================================================
# SAVE
# ============================================================
artifact = {
    'model':        final_model,
    'feature_cols': FEATURE_COLS,
    'snapshots':    sorted(df['days_into_course'].unique()),
    'cv_results':   res_df.to_dict(),
    'day_results':  {d: pd.DataFrame(v).mean().to_dict()
                     for d, v in day_results.items()},
    'removed_cols': REMOVE_COLS,
}

joblib.dump(artifact, 'edupredict_data/temporal_lgbm_model_final.pkl')
print("\n✅ Saved: edupredict_data/temporal_lgbm_model_final.pkl")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 75)
print("  v1 → v2 → final comparison")
print("=" * 75)
print(f"  {'Version':<10} | {'F1':>7} | {'AUC':>7} | Notes")
print(f"  {'-'*10}-+-{'-'*7}-+-{'-'*7}-+-{'-'*30}")
print(f"  {'v1':<10} | 0.7464  | 0.8232  | snapshots [0,14,28,56]")
print(f"  {'v2 (yours)':<10} | 0.7969  | 0.8822  | + trends, consistency ✅")
print(f"  {'v3':<10} | 0.7974  | 0.8826  | + leakage fix, iter fix")
print(f"  {'final':<10} | {res_df['f1'].mean():.4f}  | {res_df['auc'].mean():.4f}  | "
      f"+ removed days_to_start")
print(f"\n  Real behavioral F1 progression by day:")
for day in sorted(df['days_into_course'].unique()):
    avg = pd.DataFrame(day_results[day]).mean()
    bar = '█' * int(avg['f1'] * 20)
    print(f"  Day {day:>3}: {avg['f1']:.4f}  {bar}")
print("\n✅ Model ready — next: step3_realtime_predictor.py")

Loading temporal dataset v2...
Shape: (162965, 35)
Snapshots: [0, 30, 60, 90, 150]

Removed: ['days_to_start', 'registered_early']
Shape after removal: (162965, 33)

Feature count: 31
Features: ['code_module', 'gender_bin', 'disability_bin', 'age_numeric', 'edu_numeric', 'imd_numeric', 'num_of_prev_attempts', 'studied_credits', 'module_total_assessments', 'course_length', 'days_into_course', 'snapshot_progress', 'click_trend', 'total_clicks', 'active_days', 'avg_clicks_per_day', 'quiz_clicks', 'forumng_clicks', 'resource_clicks', 'clicked_before_start', 'days_since_last_click', 'click_consistency', 'avg_score', 'num_submitted', 'num_failed', 'submission_rate', 'avg_tma_score', 'avg_cma_score', 'avg_days_late', 'score_trend', 'submission_consistency']

StratifiedGroupKFold (5 folds)...

Training...
Fold 1 | Acc: 0.7949 | F1: 0.7960 | Recall: 0.7690 | Precision: 0.8249 | AUC: 0.8856 | Iters: 500
Fold 2 | Acc: 0.7990 | F1: 0.8028 | Recall: 0.7719 | Precision: 0.8363 | AUC: 0.8852 | Iters:

In [3]:
"""
STEP 3 — Real-Time Predictor
==============================
Input  : student_id + current day + live VLE/assessment logs
Output : risk_probability, risk_level, explanation, data_completeness

Compatible with: temporal_lgbm_model_final.pkl
"""

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD MODEL
# ============================================================
artifact    = joblib.load('edupredict_data/temporal_lgbm_model_final.pkl')
model       = artifact['model']
FEAT_COLS   = artifact['feature_cols']
SNAPSHOTS   = artifact['snapshots']   # [0, 30, 60, 90, 150]
DAY_RESULTS = artifact['day_results'] # per-day CV performance

print(f"Model loaded — features: {len(FEAT_COLS)}")
print(f"Snapshots trained on: {SNAPSHOTS}")

# ============================================================
# FEATURE BUILDER
# Replicates exactly what step1 did, but for ONE student NOW
# Returns NaN for features not yet available — LightGBM handles
# ============================================================

def build_vle_features(day_cutoff: int,
                       vle_log: pd.DataFrame) -> dict:
    """
    Compute VLE features from live LMS log up to day_cutoff.
    vle_log columns: date, sum_click, activity_type
    Returns NaN dict if no interactions yet.
    """
    if len(vle_log) == 0:
        return {k: np.nan for k in [
            'total_clicks','active_days','avg_clicks_per_day',
            'quiz_clicks','forumng_clicks','resource_clicks',
            'clicked_before_start','days_since_last_click',
            'click_trend','click_consistency'
        ]}

    sv = vle_log[vle_log['date'] <= day_cutoff]
    if len(sv) == 0:
        return {k: np.nan for k in [
            'total_clicks','active_days','avg_clicks_per_day',
            'quiz_clicks','forumng_clicks','resource_clicks',
            'clicked_before_start','days_since_last_click',
            'click_trend','click_consistency'
        ]}

    tc  = float(sv['sum_click'].sum())
    ad  = float(sv['date'].nunique())

    def tc_type(t):
        m = sv['activity_type'] == t
        return float(sv.loc[m,'sum_click'].sum()) if m.any() else 0.0

    # days_since_last_click
    last_click_day = float(sv['date'].max())
    days_since     = float(day_cutoff - last_click_day)

    # click_trend: clicks in last half vs first half of period
    if day_cutoff > 0:
        mid  = day_cutoff / 2
        early_clicks = float(sv[sv['date'] <= mid]['sum_click'].sum())
        late_clicks  = float(sv[sv['date'] >  mid]['sum_click'].sum())
        click_trend  = late_clicks - early_clicks
    else:
        click_trend = np.nan

    # click_consistency: active_days / days_into_course (0-1)
    click_consistency = (ad / day_cutoff) if day_cutoff > 0 else np.nan

    return {
        'total_clicks':         tc,
        'active_days':          ad,
        'avg_clicks_per_day':   round(tc / ad, 2) if ad > 0 else np.nan,
        'quiz_clicks':          tc_type('quiz'),
        'forumng_clicks':       tc_type('forumng'),
        'resource_clicks':      tc_type('resource'),
        'clicked_before_start': 1.0 if (sv['date'] < 0).any() else 0.0,
        'days_since_last_click': days_since,
        'click_trend':          click_trend,
        'click_consistency':    click_consistency,
    }


def build_assessment_features(day_cutoff: int,
                               assess_log: pd.DataFrame,
                               module_total: int) -> dict:
    """
    Compute assessment features from live LMS log up to day_cutoff.
    assess_log columns: date_submitted, score, assessment_type, date (due date)
    Returns NaN dict if no submissions yet.
    """
    nan_dict = {k: np.nan for k in [
        'avg_score','num_submitted','num_failed','submission_rate',
        'avg_tma_score','avg_cma_score','avg_days_late',
        'score_trend','submission_consistency'
    ]}

    if len(assess_log) == 0:
        return nan_dict

    sa = assess_log[assess_log['date_submitted'] <= day_cutoff]
    if len(sa) == 0:
        return nan_dict

    scores    = sa['score'].fillna(0)
    num_sub   = len(sa)
    tma       = sa[sa['assessment_type']=='TMA']['score'].dropna()
    cma       = sa[sa['assessment_type']=='CMA']['score'].dropna()
    sa_dated  = sa.dropna(subset=['date'])
    avg_late  = float((sa_dated['date_submitted'] - sa_dated['date']).mean()) \
                if len(sa_dated) > 0 else np.nan

    # score_trend: avg score last half vs first half
    if num_sub >= 2:
        half      = num_sub // 2
        sorted_sa = sa.sort_values('date_submitted')
        early_avg = sorted_sa.iloc[:half]['score'].fillna(0).mean()
        late_avg  = sorted_sa.iloc[half:]['score'].fillna(0).mean()
        score_trend = float(late_avg - early_avg)
    else:
        score_trend = np.nan

    # submission_consistency: submitted on time vs total submitted
    if len(sa_dated) > 0:
        on_time = (sa_dated['date_submitted'] <= sa_dated['date']).sum()
        sub_consistency = float(on_time / len(sa_dated))
    else:
        sub_consistency = np.nan

    return {
        'avg_score':             float(scores.mean()),
        'num_submitted':         float(num_sub),
        'num_failed':            float((scores < 40).sum()),
        'submission_rate':       min(num_sub / module_total, 1.0) \
                                 if module_total > 0 else np.nan,
        'avg_tma_score':         float(tma.mean()) if len(tma) > 0 else np.nan,
        'avg_cma_score':         float(cma.mean()) if len(cma) > 0 else np.nan,
        'avg_days_late':         avg_late,
        'score_trend':           score_trend,
        'submission_consistency': sub_consistency,
    }


# ============================================================
# MAIN PREDICT FUNCTION
# ============================================================

def predict_risk(
    day_of_course:   int,
    demographics:    dict,
    vle_log:         pd.DataFrame,
    assess_log:      pd.DataFrame,
) -> dict:
    """
    Predict at-risk probability for one student at a specific day.

    Parameters
    ----------
    day_of_course : int
        Current day since module start (0 = registration day)
    demographics  : dict
        Static student info — code_module, gender_bin, edu_numeric,
        imd_numeric, disability_bin, age_numeric, num_of_prev_attempts,
        studied_credits, module_total_assessments, course_length
    vle_log       : pd.DataFrame
        All VLE events so far — columns: date, sum_click, activity_type
    assess_log    : pd.DataFrame
        All submissions so far — columns: date_submitted, score,
        assessment_type, date (due date)

    Returns
    -------
    dict with risk_probability, risk_level, explanation, data_completeness
    """
    module_total = demographics.get('module_total_assessments', 1)
    course_len   = demographics.get('course_length', 255)

    # Behavioral features
    vle_feats    = build_vle_features(day_of_course, vle_log)
    assess_feats = build_assessment_features(
        day_of_course, assess_log, module_total
    )

    # snapshot_progress — normalize day by course length
    snapshot_progress = (day_of_course / course_len) if course_len > 0 else 0.0

    # Assemble full feature row
    row = {
        **demographics,
        'days_into_course':  day_of_course,
        'snapshot_progress': snapshot_progress,
        **vle_feats,
        **assess_feats,
    }

    # Build DataFrame aligned to training columns
    df_row = pd.DataFrame([row])
    df_row['code_module'] = df_row['code_module'].astype('category')

    # Add any missing columns as NaN
    for col in FEAT_COLS:
        if col not in df_row.columns:
            df_row[col] = np.nan

    df_row = df_row[FEAT_COLS]

    # Predict
    prob  = float(model.predict_proba(df_row)[0][1])
    label = int(prob >= 0.5)

    # Risk level — 3 tiers
    if prob >= 0.70:
        risk_level = 'HIGH'
        action     = 'Contact student immediately'
    elif prob >= 0.45:
        risk_level = 'MEDIUM'
        action     = 'Monitor closely, send reminder'
    else:
        risk_level = 'LOW'
        action     = 'No action needed'

    # Data completeness — what does the model have to work with?
    has_vle    = not np.isnan(vle_feats['total_clicks'])
    has_scores = not np.isnan(assess_feats['avg_score'])
    n_features_available = sum(
        1 for col in FEAT_COLS
        if not pd.isna(df_row[col].values[0])
        and df_row[col].dtype != 'category'
    )

    # Expected model accuracy at this day (from CV results)
    closest_day = min(SNAPSHOTS, key=lambda d: abs(d - day_of_course))
    expected_f1  = DAY_RESULTS[closest_day]['f1']
    expected_auc = DAY_RESULTS[closest_day]['auc']

    # Explanation — top reasons (based on feature importance)
    explanation = _build_explanation(df_row, assess_feats, vle_feats, prob)

    return {
        'risk_probability':  round(prob, 4),
        'risk_level':        risk_level,
        'at_risk':           label,
        'recommended_action': action,
        'explanation':       explanation,
        'model_confidence': {
            'day':           day_of_course,
            'closest_snapshot': closest_day,
            'expected_f1':   round(expected_f1, 4),
            'expected_auc':  round(expected_auc, 4),
        },
        'data_completeness': {
            'has_vle_data':        has_vle,
            'has_assessment_data': has_scores,
            'features_available':  n_features_available,
            'features_total':      len(FEAT_COLS),
            'completeness_pct':    round(n_features_available/len(FEAT_COLS)*100, 1),
        }
    }


def _build_explanation(df_row, assess_feats, vle_feats, prob):
    """Generate human-readable explanation of prediction."""
    reasons = []

    # Assessment signals
    avg_score = assess_feats.get('avg_score')
    if avg_score is not None and not np.isnan(avg_score):
        if avg_score < 40:
            reasons.append(f"Very low scores (avg={avg_score:.1f})")
        elif avg_score < 55:
            reasons.append(f"Below-average scores (avg={avg_score:.1f})")

    avg_late = assess_feats.get('avg_days_late')
    if avg_late is not None and not np.isnan(avg_late) and avg_late > 3:
        reasons.append(f"Submitting late on average ({avg_late:.1f} days)")

    score_trend = assess_feats.get('score_trend')
    if score_trend is not None and not np.isnan(score_trend) and score_trend < -10:
        reasons.append(f"Score declining ({score_trend:.1f} points)")

    # VLE signals
    tc = vle_feats.get('total_clicks')
    if tc is None or np.isnan(tc):
        reasons.append("No VLE activity recorded yet")
    elif tc < 50:
        reasons.append(f"Very low platform engagement ({int(tc)} clicks)")

    days_since = vle_feats.get('days_since_last_click')
    if days_since is not None and not np.isnan(days_since) and days_since > 14:
        reasons.append(f"No activity for {int(days_since)} days")

    click_trend = vle_feats.get('click_trend')
    if click_trend is not None and not np.isnan(click_trend) and click_trend < -100:
        reasons.append("Engagement dropping significantly")

    if not reasons:
        if prob < 0.45:
            reasons.append("Good engagement and assessment performance")
        else:
            reasons.append("Moderate risk based on combined signals")

    return reasons


# ============================================================
# BATCH PREDICTION — for all students at a given day
# ============================================================

def predict_all_students(
    day_of_course: int,
    students_df:   pd.DataFrame,
    vle_log_full:  pd.DataFrame,
    assess_log_full: pd.DataFrame,
) -> pd.DataFrame:
    """
    Run prediction for all students at once.
    Efficient: pre-filters logs per student.

    Parameters
    ----------
    students_df     : DataFrame with one row per student (demographics)
    vle_log_full    : all VLE events (must have id_student column)
    assess_log_full : all assessment submissions (must have id_student column)
    """
    vle_grouped    = {sid: grp for sid, grp in
                      vle_log_full.groupby('id_student')}
    assess_grouped = {sid: grp for sid, grp in
                      assess_log_full.groupby('id_student')}

    results = []
    for _, student in students_df.iterrows():
        sid  = student['id_student']
        demo = student.drop('id_student').to_dict()

        sv = vle_grouped.get(   sid, pd.DataFrame(columns=vle_log_full.columns))
        sa = assess_grouped.get(sid, pd.DataFrame(columns=assess_log_full.columns))

        pred = predict_risk(day_of_course, demo, sv, sa)
        results.append({
            'id_student':       sid,
            'day':              day_of_course,
            'risk_probability': pred['risk_probability'],
            'risk_level':       pred['risk_level'],
            'completeness_pct': pred['data_completeness']['completeness_pct'],
        })

    return pd.DataFrame(results).sort_values(
        'risk_probability', ascending=False
    )


# ============================================================
# DEMO — simulate a real-time scenario
# ============================================================
if __name__ == '__main__':
    print("\n" + "=" * 60)
    print("  Real-Time Predictor — Demo")
    print("=" * 60)

    # Student demographics (from enrollment system)
    demo = {
        'code_module':              'FFF',
        'gender_bin':               1,
        'disability_bin':           0,
        'age_numeric':              0,
        'edu_numeric':              2,
        'imd_numeric':              40.0,
        'num_of_prev_attempts':     0,
        'studied_credits':          60,
        'module_total_assessments': 13,
        'course_length':            255,
    }

    # Simulated VLE log — student was active early, then went quiet
    vle_log = pd.DataFrame({
        'date':          [-5, 2, 5, 8, 12, 15, 18, 20],
        'sum_click':     [4, 20, 15, 8, 25, 12, 3, 1],
        'activity_type': ['resource','forumng','quiz','resource',
                          'forumng','quiz','resource','resource']
    })

    # Simulated assessment log — one submission, slightly late
    assess_log = pd.DataFrame({
        'date_submitted':  [22],
        'score':           [58.0],
        'assessment_type': ['TMA'],
        'date':            [19.0],   # due date
    })

    # Predict at different time points
    print(f"\n{'Day':>5} | {'Prob':>6} | {'Level':>6} | "
          f"{'F1 expected':>12} | {'Data%':>6} | Explanation")
    print("-" * 75)

    for day in [0, 30, 60, 90, 150]:
        result = predict_risk(day, demo, vle_log, assess_log)
        exp    = result['explanation'][0] if result['explanation'] else ''
        conf   = result['model_confidence']
        comp   = result['data_completeness']

        print(f"{day:>5} | {result['risk_probability']:>6.3f} | "
              f"{result['risk_level']:>6} | "
              f"{conf['expected_f1']:>12.4f} | "
              f"{comp['completeness_pct']:>5.1f}% | {exp}")

    print("\n--- Full prediction at Day 60 ---")
    result = predict_risk(60, demo, vle_log, assess_log)
    print(f"Risk probability : {result['risk_probability']}")
    print(f"Risk level       : {result['risk_level']}")
    print(f"Action           : {result['recommended_action']}")
    print(f"Reasons          : {result['explanation']}")
    print(f"Model confidence : F1={result['model_confidence']['expected_f1']} "
          f"AUC={result['model_confidence']['expected_auc']}")
    print(f"Data completeness: {result['data_completeness']['completeness_pct']}% "
          f"({result['data_completeness']['features_available']}/"
          f"{result['data_completeness']['features_total']} features)")

Model loaded — features: 31
Snapshots trained on: [0, 30, 60, 90, 150]

  Real-Time Predictor — Demo

  Day |   Prob |  Level |  F1 expected |  Data% | Explanation
---------------------------------------------------------------------------
    0 |  0.480 | MEDIUM |       0.7075 |  61.3% | Very low platform engagement (4 clicks)
   30 |  0.908 |   HIGH |       0.7624 |  90.3% | Moderate risk based on combined signals
   60 |  0.993 |   HIGH |       0.8083 |  90.3% | No activity for 40 days
   90 |  0.996 |   HIGH |       0.8338 |  90.3% | No activity for 70 days
  150 |  0.999 |   HIGH |       0.8812 |  90.3% | No activity for 130 days

--- Full prediction at Day 60 ---
Risk probability : 0.9933
Risk level       : HIGH
Action           : Contact student immediately
Reasons          : ['No activity for 40 days']
Model confidence : F1=0.8083 AUC=0.891
Data completeness: 90.3% (28/31 features)


In [4]:
"""
STEP 4 — Final Evaluation on Holdout Test Set
===============================================
This is the ONLY time we touch test data.
Evaluates the temporal model on unseen students.

Input  : temporal_lgbm_model_final.pkl
         temporal_dataset_v2.csv (to rebuild test snapshots)
         edupredict_data/test.csv (original holdout — for student IDs)
Output : final evaluation report
"""

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, accuracy_score, confusion_matrix,
    classification_report
)
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD
# ============================================================
print("Loading model and data...")
artifact  = joblib.load('edupredict_data/temporal_lgbm_model_final.pkl')
model     = artifact['model']
FEAT_COLS = artifact['feature_cols']
SNAPSHOTS = artifact['snapshots']

# Load temporal dataset — split into train/test by original student split
temporal  = pd.read_csv('edupredict_data/temporal_dataset_v2.csv')
original_test = pd.read_csv('edupredict_data/test.csv')

# Get test student IDs from the original holdout split
test_ids  = set(original_test['id_student'].unique()) \
            if 'id_student' in original_test.columns \
            else None

# ============================================================
# REBUILD TEST SET
# If test.csv has id_student → filter temporal by those IDs
# Otherwise → use 20% of temporal (same stratified split)
# ============================================================
if test_ids:
    test_temporal = temporal[temporal['id_student'].isin(test_ids)].copy()
    print(f"Test students from original holdout: {len(test_ids)}")
else:
    # Fallback: recreate 80/20 split by student
    from sklearn.model_selection import train_test_split
    all_students = temporal['id_student'].unique()
    labels       = temporal.groupby('id_student')['at_risk'].first()
    _, test_studs = train_test_split(
        all_students, test_size=0.2,
        stratify=labels[all_students],
        random_state=42
    )
    test_temporal = temporal[temporal['id_student'].isin(test_studs)].copy()
    print(f"Test students (recreated 20% split): {len(test_studs)}")

print(f"Test snapshots total: {len(test_temporal):,}")

# Remove same columns as training
REMOVE_COLS = ['days_to_start', 'registered_early', 'late_reg_x_progress']
REMOVE_COLS = [c for c in REMOVE_COLS if c in test_temporal.columns]
test_temporal = test_temporal.drop(columns=REMOVE_COLS)

# ============================================================
# PREPARE FEATURES
# ============================================================
X_test = test_temporal[FEAT_COLS].copy()
y_test = test_temporal['at_risk'].copy()

X_test['code_module'] = X_test['code_module'].astype('category')

# ============================================================
# PREDICT
# ============================================================
print("Running predictions...")
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# ============================================================
# OVERALL RESULTS
# ============================================================
acc  = accuracy_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\n" + "=" * 60)
print("  EDUPREDICT TEMPORAL — Final Test Set Results")
print("=" * 60)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  F1-Score  : {f1*100:.2f}%")
print(f"  Recall    : {rec*100:.2f}%")
print(f"  Precision : {prec*100:.2f}%")
print(f"  AUC       : {auc*100:.2f}%")
print("=" * 60)

print(f"\nConfusion Matrix:")
print(f"                    Predicted")
print(f"                  Safe    Risk")
print(f"Actual  Safe  [{tn:5,}  {fp:5,}]")
print(f"        Risk  [{fn:5,}  {tp:5,}]")

print(f"\nKey numbers:")
print(f"  At-risk students correctly flagged : {tp:,}  ({tp/(tp+fn)*100:.1f}%)")
print(f"  At-risk students MISSED            : {fn:,}  ({fn/(tp+fn)*100:.1f}%)")
print(f"  False alarms                       : {fp:,}  ({fp/(tn+fp)*100:.1f}%)")

# ============================================================
# PER-DAY RESULTS ON TEST SET
# ============================================================
print("\n" + "=" * 60)
print("  Per-Day Performance on Test Set (unseen students)")
print("=" * 60)
print(f"{'Day':>5} | {'N':>5} | {'F1':>6} | {'Recall':>6} | "
      f"{'Precision':>9} | {'AUC':>6} | {'Missed':>6}")
print("-" * 60)

test_temporal['y_pred'] = y_pred
test_temporal['y_prob'] = y_prob

day_rows = []
for day in sorted(test_temporal['days_into_course'].unique()):
    sub = test_temporal[test_temporal['days_into_course'] == day]
    f1_d   = f1_score(sub['at_risk'], sub['y_pred'])
    rec_d  = recall_score(sub['at_risk'], sub['y_pred'])
    prec_d = precision_score(sub['at_risk'], sub['y_pred'])
    auc_d  = roc_auc_score(sub['at_risk'], sub['y_prob'])
    cm_d   = confusion_matrix(sub['at_risk'], sub['y_pred'])
    fn_d   = cm_d[1][0]   # at-risk students missed
    day_rows.append({
        'day': day, 'f1': f1_d, 'recall': rec_d,
        'precision': prec_d, 'auc': auc_d, 'missed': fn_d
    })
    print(f"{day:>5} | {len(sub):>5,} | {f1_d:.4f} | {rec_d:.4f} | "
          f"{prec_d:.9f} | {auc_d:.4f} | {fn_d:>6,}")

# ============================================================
# COMPARISON: OLD MODEL vs NEW MODEL
# ============================================================
print("\n" + "=" * 60)
print("  Old Model vs Temporal Model")
print("=" * 60)
print(f"  {'Metric':<12} | {'Old Model':>10} | {'Temporal':>10} | {'Change':>8}")
print(f"  {'-'*12}-+-{'-'*10}-+-{'-'*10}-+-{'-'*8}")

old = {'Accuracy': 0.9184, 'F1': 0.9204, 'Recall': 0.8937,
       'Precision': 0.9488, 'AUC': 0.9742}
new = {'Accuracy': acc, 'F1': f1, 'Recall': rec,
       'Precision': prec, 'AUC': auc}

for metric in ['Accuracy', 'F1', 'Recall', 'Precision', 'AUC']:
    diff  = new[metric] - old[metric]
    arrow = '↑' if diff > 0 else '↓'
    print(f"  {metric:<12} | {old[metric]:>10.4f} | {new[metric]:>10.4f} | "
          f"{arrow}{abs(diff):.4f}")

print(f"""
  Important context:
  - Old model: trained on END-OF-COURSE data → not real-time capable
  - Temporal:  trained on snapshots → works at any point in course

  The old 91.8% accuracy is NOT comparable — it had access to
  final scores and total VLE clicks, which are future information.
  The temporal model's numbers are honest real-time performance.
""")

# ============================================================
# WHAT DOES 79.8% F1 MEAN IN PRACTICE?
# ============================================================
total_at_risk = tp + fn
print("=" * 60)
print("  What does this mean for instructors?")
print("=" * 60)
print(f"  In a cohort of {len(sub):,} students per snapshot day:")
print(f"  - Model correctly warns about : {tp:,} at-risk students")
print(f"  - Model misses                : {fn:,} at-risk students")
print(f"  - False alarms per day        : {fp:,} students")
print(f"\n  At Day 150 (most data available):")
sub150 = test_temporal[test_temporal['days_into_course'] == 150]
cm150  = confusion_matrix(sub150['at_risk'], sub150['y_pred'])
tn150, fp150, fn150, tp150 = cm150.ravel()
rec150 = tp150 / (tp150 + fn150)
print(f"  - Recall = {rec150*100:.1f}%: catches {tp150:,} of {tp150+fn150:,} at-risk students")
print(f"  - Misses {fn150:,} students who will fail/withdraw")
print(f"  - False alarms: {fp150:,} students incorrectly flagged")

Loading model and data...
Test students (recreated 20% split): 5757
Test snapshots total: 32,635
Running predictions...

  EDUPREDICT TEMPORAL — Final Test Set Results
  Accuracy  : 82.56%
  F1-Score  : 82.84%
  Recall    : 80.10%
  Precision : 85.78%
  AUC       : 91.34%

Confusion Matrix:
                    Predicted
                  Safe    Risk
Actual  Safe  [13,207  2,278]
        Risk  [3,413  13,737]

Key numbers:
  At-risk students correctly flagged : 13,737  (80.1%)
  At-risk students MISSED            : 3,413  (19.9%)
  False alarms                       : 2,278  (14.7%)

  Per-Day Performance on Test Set (unseen students)
  Day |     N |     F1 | Recall | Precision |    AUC | Missed
------------------------------------------------------------
    0 | 6,527 | 0.7330 | 0.7402 | 0.725843339 | 0.7870 |    891
   30 | 6,527 | 0.7978 | 0.7580 | 0.841968912 | 0.8879 |    830
   60 | 6,527 | 0.8429 | 0.8035 | 0.886458668 | 0.9256 |    674
   90 | 6,527 | 0.8679 | 0.8338 | 0.904776

In [1]:
"""
STEP 5 — Threshold Tuning
==========================
Default threshold = 0.5 treats both errors equally.
In early warning systems: missing a at-risk student (FN)
is MORE costly than a false alarm (FP).

We find the optimal threshold that minimizes:
  cost = w_fn * FN + w_fp * FP

where w_fn > w_fp (missing at-risk = worse)
"""

import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD
# ============================================================
print("Loading model and data...")
artifact  = joblib.load('edupredict_data/temporal_lgbm_model_final.pkl')
model     = artifact['model']
FEAT_COLS = artifact['feature_cols']

temporal  = pd.read_csv('edupredict_data/temporal_dataset_v2.csv')

# Recreate same 20% test split
from sklearn.model_selection import train_test_split
all_students = temporal['id_student'].unique()
labels       = temporal.groupby('id_student')['at_risk'].first()
_, test_studs = train_test_split(
    all_students, test_size=0.2,
    stratify=labels[all_students], random_state=42
)
test_df = temporal[temporal['id_student'].isin(test_studs)].copy()

REMOVE = ['days_to_start','registered_early','late_reg_x_progress']
test_df = test_df.drop(columns=[c for c in REMOVE if c in test_df.columns])

X_test = test_df[FEAT_COLS].copy()
y_test = test_df['at_risk'].copy()
X_test['code_module'] = X_test['code_module'].astype('category')

y_prob = model.predict_proba(X_test)[:, 1]

# ============================================================
# STEP A — Scan all thresholds
# ============================================================
print("\nScanning thresholds 0.10 → 0.90...")
thresholds = np.arange(0.10, 0.91, 0.01)

rows = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    cm     = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    rows.append({
        'threshold':  round(t, 2),
        'f1':         f1_score(y_test, y_pred),
        'recall':     recall_score(y_test, y_pred),
        'precision':  precision_score(y_test, y_pred),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'missed':     fn,
        'false_alarms': fp,
    })

scan = pd.DataFrame(rows)

# ============================================================
# STEP B — Cost-based optimal threshold
# FN cost = 3x FP cost (missing at-risk student = 3x worse)
# Adjust this ratio based on institutional preference
# ============================================================
W_FN = 3   # cost of missing one at-risk student
W_FP = 1   # cost of one false alarm

scan['cost'] = W_FN * scan['fn'] + W_FP * scan['fp']
scan['cost_normalized'] = scan['cost'] / scan['cost'].max()

idx_cost  = scan['cost'].idxmin()
idx_f1    = scan['f1'].idxmax()
idx_rec   = (scan['recall'] >= 0.90).idxmax()   # first threshold with recall ≥ 90%

t_cost  = scan.loc[idx_cost,  'threshold']
t_f1    = scan.loc[idx_f1,    'threshold']
t_90rec = scan.loc[idx_rec,   'threshold'] if scan['recall'].max() >= 0.90 else None

# ============================================================
# STEP C — Results
# ============================================================
print("\n" + "=" * 65)
print("  Threshold Analysis Results")
print("=" * 65)

def print_threshold(label, t, df):
    row = df[df['threshold'] == round(t, 2)].iloc[0]
    total_risk = row['tp'] + row['fn']
    print(f"\n  {label} (threshold = {t})")
    print(f"  {'─'*40}")
    print(f"  F1        : {row['f1']*100:.2f}%")
    print(f"  Recall    : {row['recall']*100:.2f}%")
    print(f"  Precision : {row['precision']*100:.2f}%")
    print(f"  Missed    : {int(row['missed']):,}  "
          f"({row['missed']/total_risk*100:.1f}% of at-risk students)")
    print(f"  False alarms: {int(row['false_alarms']):,}")
    print(f"  Cost score: {row['cost']:,.0f}  "
          f"(fn×{W_FN} + fp×{W_FP})")

print_threshold("Default    t=0.50", 0.50, scan)
print_threshold("Best F1   ", t_f1,   scan)
print_threshold("Best cost ", t_cost, scan)
if t_90rec:
    print_threshold("90% recall", t_90rec, scan)

# ============================================================
# STEP D — Per-day analysis at optimal threshold
# ============================================================
print("\n" + "=" * 65)
print(f"  Per-day results at optimal threshold (t={t_cost})")
print("=" * 65)
print(f"{'Day':>5} | {'F1':>6} | {'Recall':>6} | {'Precision':>9} | "
      f"{'Missed':>6} | {'FAlarms':>7}")
print("-" * 55)

test_df['y_prob'] = y_prob

for day in sorted(test_df['days_into_course'].unique()):
    sub    = test_df[test_df['days_into_course'] == day]
    y_d    = sub['at_risk']
    p_d    = sub['y_prob']
    pred_d = (p_d >= t_cost).astype(int)
    cm_d   = confusion_matrix(y_d, pred_d)
    tn_d, fp_d, fn_d, tp_d = cm_d.ravel()
    print(f"{day:>5} | "
          f"{f1_score(y_d,pred_d):>6.4f} | "
          f"{recall_score(y_d,pred_d):>6.4f} | "
          f"{precision_score(y_d,pred_d):>9.4f} | "
          f"{fn_d:>6,} | {fp_d:>7,}")

# ============================================================
# STEP E — Save plot
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Threshold Tuning Analysis', fontsize=14)

# Plot 1: F1, Recall, Precision vs threshold
ax = axes[0]
ax.plot(scan['threshold'], scan['f1'],        label='F1',        color='#1D9E75')
ax.plot(scan['threshold'], scan['recall'],    label='Recall',    color='#378ADD')
ax.plot(scan['threshold'], scan['precision'], label='Precision', color='#EF9F27')
ax.axvline(0.50,   color='gray',    linestyle='--', alpha=0.5, label='default 0.5')
ax.axvline(t_cost, color='#D85A30', linestyle='-',  alpha=0.8, label=f'optimal {t_cost}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Metrics vs Threshold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Plot 2: Cost vs threshold
ax = axes[1]
ax.plot(scan['threshold'], scan['cost'], color='#D85A30')
ax.axvline(t_cost, color='#D85A30', linestyle='--', alpha=0.8,
           label=f'min cost @ {t_cost}')
ax.axvline(0.50,   color='gray',    linestyle='--', alpha=0.5,
           label='default 0.5')
ax.set_xlabel('Threshold')
ax.set_ylabel(f'Cost (FN×{W_FN} + FP×{W_FP})')
ax.set_title('Cost Function')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Plot 3: Missed vs False Alarms tradeoff
ax = axes[2]
ax.plot(scan['false_alarms'], scan['missed'],
        color='#1D9E75', linewidth=2)
# Mark key points
for t_val, label, color in [
    (0.50,   'default', 'gray'),
    (t_cost, f'optimal\nt={t_cost}', '#D85A30'),
]:
    row = scan[scan['threshold'] == round(t_val, 2)].iloc[0]
    ax.scatter(row['false_alarms'], row['missed'],
               color=color, s=80, zorder=5)
    ax.annotate(label,
                xy=(row['false_alarms'], row['missed']),
                xytext=(10, 5), textcoords='offset points',
                fontsize=8, color=color)
ax.set_xlabel('False Alarms (FP)')
ax.set_ylabel('Missed At-Risk Students (FN)')
ax.set_title('FP vs FN Tradeoff')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('edupredict_data/threshold_analysis.png', dpi=150, bbox_inches='tight')
print(f"\nPlot saved: edupredict_data/threshold_analysis.png")

# ============================================================
# STEP F — Save optimal threshold to artifact
# ============================================================
artifact['optimal_threshold'] = t_cost
artifact['threshold_scan']    = scan.to_dict()
joblib.dump(artifact, 'edupredict_data/temporal_lgbm_model_final.pkl')

print("\n" + "=" * 65)
print("  Summary — Which threshold to use?")
print("=" * 65)

row_def  = scan[scan['threshold'] == 0.50].iloc[0]
row_opt  = scan[scan['threshold'] == round(t_cost, 2)].iloc[0]
total_risk = row_def['tp'] + row_def['fn']
saved    = int(row_def['missed'] - row_opt['missed'])
extra_fp = int(row_opt['false_alarms'] - row_def['false_alarms'])

print(f"""
  Default (t=0.50):
    Missed : {int(row_def['missed']):,} at-risk students
    FAlarms: {int(row_def['false_alarms']):,} false alarms

  Optimal (t={t_cost}):
    Missed : {int(row_opt['missed']):,} at-risk students  ← {saved:,} fewer missed
    FAlarms: {int(row_opt['false_alarms']):,} false alarms  ← {extra_fp:,} more false alarms

  Trade-off:
    Catch {saved:,} more at-risk students
    at the cost of {extra_fp:,} extra false alarms
    → Worth it if instructors can handle {extra_fp:,} extra checks

  Optimal threshold saved to model artifact.
  Next: API deployment with this threshold baked in.
""")

Loading model and data...

Scanning thresholds 0.10 → 0.90...

  Threshold Analysis Results

  Default    t=0.50 (threshold = 0.5)
  ────────────────────────────────────────
  F1        : 82.84%
  Recall    : 80.10%
  Precision : 85.78%
  Missed    : 3,413  (19.9% of at-risk students)
  False alarms: 2,278
  Cost score: 12,517  (fn×3 + fp×1)

  Best F1    (threshold = 0.41)
  ────────────────────────────────────────
  F1        : 83.53%
  Recall    : 86.51%
  Precision : 80.75%
  Missed    : 2,313  (13.5% of at-risk students)
  False alarms: 3,537
  Cost score: 10,476  (fn×3 + fp×1)

  Best cost  (threshold = 0.27)
  ────────────────────────────────────────
  F1        : 81.25%
  Recall    : 94.30%
  Precision : 71.37%
  Missed    : 978  (5.7% of at-risk students)
  False alarms: 6,486
  Cost score: 9,420  (fn×3 + fp×1)

  90% recall (threshold = 0.1)
  ────────────────────────────────────────
  F1        : 73.60%
  Recall    : 99.62%
  Precision : 58.36%
  Missed    : 66  (0.4% of at-

In [3]:
"""
STEP 5 — Threshold Tuning
==========================
Default threshold = 0.5 treats both errors equally.
In early warning systems: missing a at-risk student (FN)
is MORE costly than a false alarm (FP).

We find the optimal threshold that minimizes:
  cost = w_fn * FN + w_fp * FP

where w_fn > w_fp (missing at-risk = worse)
"""

import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, confusion_matrix, accuracy_score   # ← أُضيف هنا
)
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD
# ============================================================
print("Loading model and data...")
artifact  = joblib.load('edupredict_data/temporal_lgbm_model_final.pkl')
model     = artifact['model']
FEAT_COLS = artifact['feature_cols']

temporal  = pd.read_csv('edupredict_data/temporal_dataset_v2.csv')

from sklearn.model_selection import train_test_split
all_students = temporal['id_student'].unique()
labels       = temporal.groupby('id_student')['at_risk'].first()
_, test_studs = train_test_split(
    all_students, test_size=0.2,
    stratify=labels[all_students], random_state=42
)
test_df = temporal[temporal['id_student'].isin(test_studs)].copy()

REMOVE = ['days_to_start','registered_early','late_reg_x_progress']
test_df = test_df.drop(columns=[c for c in REMOVE if c in test_df.columns])

X_test = test_df[FEAT_COLS].copy()
y_test = test_df['at_risk'].copy()
X_test['code_module'] = X_test['code_module'].astype('category')

y_prob = model.predict_proba(X_test)[:, 1]

# ============================================================
# STEP A — Scan all thresholds
# ============================================================
print("\nScanning thresholds 0.10 → 0.90...")
thresholds = np.arange(0.10, 0.91, 0.01)

rows = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    cm     = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    rows.append({
        'threshold':   round(t, 2),
        'accuracy':    accuracy_score(y_test, y_pred),   # ← أُضيف هنا
        'f1':          f1_score(y_test, y_pred),
        'recall':      recall_score(y_test, y_pred),
        'precision':   precision_score(y_test, y_pred),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'missed':      fn,
        'false_alarms': fp,
    })

scan = pd.DataFrame(rows)

# ============================================================
# STEP B — Cost-based optimal threshold
# ============================================================
W_FN = 3
W_FP = 1

scan['cost']            = W_FN * scan['fn'] + W_FP * scan['fp']
scan['cost_normalized'] = scan['cost'] / scan['cost'].max()

idx_cost  = scan['cost'].idxmin()
idx_f1    = scan['f1'].idxmax()
idx_rec   = (scan['recall'] >= 0.90).idxmax()

t_cost  = scan.loc[idx_cost, 'threshold']
t_f1    = scan.loc[idx_f1,   'threshold']
t_90rec = scan.loc[idx_rec,  'threshold'] if scan['recall'].max() >= 0.90 else None

# ============================================================
# STEP C — Results
# ============================================================
print("\n" + "=" * 65)
print("  Threshold Analysis Results")
print("=" * 65)

def print_threshold(label, t, df):
    row       = df[df['threshold'] == round(t, 2)].iloc[0]
    total_risk = row['tp'] + row['fn']
    print(f"\n  {label} (threshold = {t})")
    print(f"  {'─'*40}")
    print(f"  Accuracy  : {row['accuracy']*100:.2f}%")          # ← أُضيف هنا
    print(f"  F1        : {row['f1']*100:.2f}%")
    print(f"  Recall    : {row['recall']*100:.2f}%")
    print(f"  Precision : {row['precision']*100:.2f}%")
    print(f"  Missed    : {int(row['missed']):,}  "
          f"({row['missed']/total_risk*100:.1f}% of at-risk students)")
    print(f"  False alarms: {int(row['false_alarms']):,}")
    print(f"  Cost score: {row['cost']:,.0f}  "
          f"(fn×{W_FN} + fp×{W_FP})")

print_threshold("Default    t=0.50", 0.50, scan)
print_threshold("Best F1   ",        t_f1,   scan)
print_threshold("Best cost ",        t_cost, scan)
if t_90rec:
    print_threshold("90% recall",    t_90rec, scan)

# ============================================================
# STEP D — Per-day analysis at optimal threshold
# ============================================================
print("\n" + "=" * 65)
print(f"  Per-day results at optimal threshold (t={t_cost})")
print("=" * 65)
print(f"{'Day':>5} | {'Accuracy':>8} | {'F1':>6} | {'Recall':>6} | "   # ← أُضيف هنا
      f"{'Precision':>9} | {'Missed':>6} | {'FAlarms':>7}")
print("-" * 65)

test_df['y_prob'] = y_prob

for day in sorted(test_df['days_into_course'].unique()):
    sub    = test_df[test_df['days_into_course'] == day]
    y_d    = sub['at_risk']
    p_d    = sub['y_prob']
    pred_d = (p_d >= t_cost).astype(int)
    cm_d   = confusion_matrix(y_d, pred_d)
    tn_d, fp_d, fn_d, tp_d = cm_d.ravel()
    print(f"{day:>5} | "
          f"{accuracy_score(y_d, pred_d):>8.4f} | "                      # ← أُضيف هنا
          f"{f1_score(y_d, pred_d):>6.4f} | "
          f"{recall_score(y_d, pred_d):>6.4f} | "
          f"{precision_score(y_d, pred_d):>9.4f} | "
          f"{fn_d:>6,} | {fp_d:>7,}")

# ============================================================
# STEP E — Save plot  (4 subplots بدلاً من 3)
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(20, 5))          # ← تغيّر من 3 إلى 4
fig.suptitle('Threshold Tuning Analysis', fontsize=14)

# Plot 1: Accuracy vs threshold  ← جديد كلياً
ax = axes[0]
ax.plot(scan['threshold'], scan['accuracy'], color='#B4BEFE', linewidth=2)
ax.axvline(0.50,   color='gray',    linestyle='--', alpha=0.5, label='default 0.5')
ax.axvline(t_cost, color='#D85A30', linestyle='-',  alpha=0.8, label=f'optimal {t_cost}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Threshold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Plot 2: F1, Recall, Precision vs threshold
ax = axes[1]
ax.plot(scan['threshold'], scan['f1'],        label='F1',        color='#1D9E75')
ax.plot(scan['threshold'], scan['recall'],    label='Recall',    color='#378ADD')
ax.plot(scan['threshold'], scan['precision'], label='Precision', color='#EF9F27')
ax.axvline(0.50,   color='gray',    linestyle='--', alpha=0.5, label='default 0.5')
ax.axvline(t_cost, color='#D85A30', linestyle='-',  alpha=0.8, label=f'optimal {t_cost}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Metrics vs Threshold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Plot 3: Cost vs threshold
ax = axes[2]
ax.plot(scan['threshold'], scan['cost'], color='#D85A30')
ax.axvline(t_cost, color='#D85A30', linestyle='--', alpha=0.8,
           label=f'min cost @ {t_cost}')
ax.axvline(0.50,   color='gray',    linestyle='--', alpha=0.5,
           label='default 0.5')
ax.set_xlabel('Threshold')
ax.set_ylabel(f'Cost (FN×{W_FN} + FP×{W_FP})')
ax.set_title('Cost Function')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Plot 4: Missed vs False Alarms tradeoff
ax = axes[3]
ax.plot(scan['false_alarms'], scan['missed'],
        color='#1D9E75', linewidth=2)
for t_val, label, color in [
    (0.50,   'default', 'gray'),
    (t_cost, f'optimal\nt={t_cost}', '#D85A30'),
]:
    row = scan[scan['threshold'] == round(t_val, 2)].iloc[0]
    ax.scatter(row['false_alarms'], row['missed'],
               color=color, s=80, zorder=5)
    ax.annotate(label,
                xy=(row['false_alarms'], row['missed']),
                xytext=(10, 5), textcoords='offset points',
                fontsize=8, color=color)
ax.set_xlabel('False Alarms (FP)')
ax.set_ylabel('Missed At-Risk Students (FN)')
ax.set_title('FP vs FN Tradeoff')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('edupredict_data/threshold_analysis.png', dpi=150, bbox_inches='tight')
print(f"\nPlot saved: edupredict_data/threshold_analysis.png")

# ============================================================
# STEP F — Save optimal threshold to artifact
# ============================================================
artifact['optimal_threshold'] = t_cost
artifact['threshold_scan']    = scan.to_dict()
joblib.dump(artifact, 'edupredict_data/temporal_lgbm_model_final.pkl')

print("\n" + "=" * 65)
print("  Summary — Which threshold to use?")
print("=" * 65)

row_def    = scan[scan['threshold'] == 0.50].iloc[0]
row_opt    = scan[scan['threshold'] == round(t_cost, 2)].iloc[0]
total_risk = row_def['tp'] + row_def['fn']
saved      = int(row_def['missed']       - row_opt['missed'])
extra_fp   = int(row_opt['false_alarms'] - row_def['false_alarms'])

print(f"""
  Default (t=0.50):
    Accuracy: {row_def['accuracy']*100:.2f}%
    Missed  : {int(row_def['missed']):,} at-risk students
    FAlarms : {int(row_def['false_alarms']):,} false alarms

  Optimal (t={t_cost}):
    Accuracy: {row_opt['accuracy']*100:.2f}%
    Missed  : {int(row_opt['missed']):,} at-risk students  ← {saved:,} fewer missed
    FAlarms : {int(row_opt['false_alarms']):,} false alarms  ← {extra_fp:,} more false alarms

  Trade-off:
    Catch {saved:,} more at-risk students
    at the cost of {extra_fp:,} extra false alarms
    → Worth it if instructors can handle {extra_fp:,} extra checks


""")

Loading model and data...

Scanning thresholds 0.10 → 0.90...

  Threshold Analysis Results

  Default    t=0.50 (threshold = 0.5)
  ────────────────────────────────────────
  Accuracy  : 82.56%
  F1        : 82.84%
  Recall    : 80.10%
  Precision : 85.78%
  Missed    : 3,413  (19.9% of at-risk students)
  False alarms: 2,278
  Cost score: 12,517  (fn×3 + fp×1)

  Best F1    (threshold = 0.41)
  ────────────────────────────────────────
  Accuracy  : 82.07%
  F1        : 83.53%
  Recall    : 86.51%
  Precision : 80.75%
  Missed    : 2,313  (13.5% of at-risk students)
  False alarms: 3,537
  Cost score: 10,476  (fn×3 + fp×1)

  Best cost  (threshold = 0.27)
  ────────────────────────────────────────
  Accuracy  : 77.13%
  F1        : 81.25%
  Recall    : 94.30%
  Precision : 71.37%
  Missed    : 978  (5.7% of at-risk students)
  False alarms: 6,486
  Cost score: 9,420  (fn×3 + fp×1)

  90% recall (threshold = 0.1)
  ────────────────────────────────────────
  Accuracy  : 62.45%
  F1     